#Inference Optimization
In this notebook, I benchmark the basic transformers library forward pass for the TinyLlama-1.1B-Chat-v1.0 llm. I then iteratively optimize the inference via vllm and various techniques.

In [ ]:
%pip install -q transformers torch matplotlib
nvidia-smi

In [ ]:
import time
import statistics
import torch
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer

assert torch.cuda.is_available(), "GPU runtime not enabled"
print(torch.cuda.get_device_name(0))

# Load Model

In [ ]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
dtype = torch.bfloat16

tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_name, dtype=dtype, attn_implementation="sdpa").to("cuda")
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"{model_name}: {n_params/1e6:.1f}M params, dtype = {dtype}")

# Timing Latency
Here, I establish functions to manually time the prefill latency (TTFT) and the decode latency (ITL samples, mean = TPOT)

In [ ]:
@torch.no_grad()
